# Canonical Model 00 · Conceptual Model and Build

Every notebook in this set uses **one** model: an irregular DISV/Voronoi
**alluvial valley**. Two tributary streams enter the mountain front, **converge**
at a confluence, and the combined main stem discharges to a single terminal
**lake** near the valley mouth. The streams **gain** from the aquifer in the
headwaters and **lose** to it downgradient, where a cross-valley bedrock
constriction steps the water table down and the lake perches above it.

> **Purpose:** establish one trusted, package-rich model whose physics,
> observations, visual diagnostics, particle tracking, parallel splitting, and
> PEST calibration can all be exercised against the same truth.

### Four hydrostratigraphic layers

| Layer | Unit | Hydrogeologic role |
|---|---|---|
| 1 | Upper unconfined alluvium | Water table, UZF infiltration, stream & spring **seepage** |
| 2 | Lower unconfined (main) aquifer | Regional flow, shallow pumping, high-K paleochannel |
| 3 | Lacustrine clay aquitard | Distinct low-K; damps vertical communication |
| 4 | Confined basin-fill aquifer | Deep pumping cone, confined response |

### Boundary conditions
CHD (up-valley inflow) · GHB (valley-mouth outflow) · RCH (areal recharge) ·
UZF (unsaturated-zone recharge incl. an **infiltration** pond) · WEL (shallow +
deep pumping) · DRN (two aquifer **seepage** spring slopes) · SFR (the converging
stream network) · LAK (the terminal lake) · MVR (main stem → lake).

In [1]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
import myflopy as mf
from canonical_notebook_style import notebook_header

notebook_header('00', 'Conceptual Model and Build', 'One valley model, one contract, every workflow.')

workspace = Path('../artifacts/canonical_master')
config = mf.CanonicalModelConfig.validation()   # 50x50; use CanonicalModelConfig() for the full >=10,000-cell profile
model = mf.build_canonical_model(workspace / 'gwf', config=config, name='canonical_master')

# The contract checks grid type, four-layer hydrostratigraphy, the full package
# set, the named observation targets, and that surface-water cells are refined.
mf.CANONICAL_MODEL_CONTRACT.validate(model)
model.regions.region_summary()

VoronoiGrid initializing.
Voronoi grid initialized.


getting connectivity properties (iac, ja, cl12, hwva, nja)


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(


,name,kind,category,package,layer,num_cells,tags
3,all_lakes,region,boundary,lak,0,130,[lak]
4,all_streams,region,boundary,sfr,0,128,[sfr]
6,infiltration_pond,region,infiltration,uzf,0,3,"[mounding, visualization]"
2,lake_zone_valley_lake,region,boundary,lak,0,130,[lak]
0,north_seepage_springs,region,seepage,drn,0,28,[]
1,south_seepage_springs,region,seepage,drn,0,27,[]
5,uzf_active,region,boundary,uzf,0,1322,[uzf]


## What the build contains

The `region_summary` above lists the named feature regions the rest of the set
queries by name: `north_seepage_springs` / `south_seepage_springs` (DRN slopes),
`infiltration_pond` (UZF), `all_streams` (the converging SFR network),
`all_lakes` / `lake_zone_valley_lake` (the terminal lake), and `uzf_active`
(the valley floor). Surface-water cells are deliberately **refined** so the
stream corridor and lake resolve on finer Voronoi cells.

In [2]:
success, report = model.run_simulation()
assert success, '\n'.join(report[-30:])

# Head signals: a strong regional gradient in every layer, transient movement in
# the unconfined aquifer, and a distinct (smaller) confined response.
mf.canonical_head_signals(model)

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model canonical_master...
    writing model name file...
    writing package disv...
    writing package ic...


    writing package npf...
    writing package sto...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 200 based on size of stress_period_data
    writing package ghb...
INFORMATION: maxbound in ('', 'ghb', 'dimensions') changed to 200 based on size of stress_period_data
    writing package rch...
    writing package wel...
INFORMATION: maxbound in ('', 'wel', 'dimensions') changed to 2 based on size of stress_period_data


    writing package drn...
INFORMATION: maxbound in ('', 'drn', 'dimensions') changed to 55 based on size of stress_period_data
    writing package lak...
    writing package sfr...
    writing package mvr...
    writing package uzf...


    writing package gwf_obs...
    writing package lak_obs...
    writing package sfr_obs...
    writing package drn_flow_obs...
    writing package oc...

Saved model object to .model file: ..\artifacts\canonical_master\gwf\canonical_master\canonical_master.model

FloPy is using the following executable to run the model: ..\..\..\..\..\..\..\..\PATH\modflow_exe\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.7.0 02/05/2026

   MODFLOW 6 compiled Feb 05 2026 22:36:44 with Intel(R) Fortran Intel(R) 64
   Compiler Classic for applications running on Intel(R) 64, Version 2021.6.0
                             Build 20220226_000000

This software has been approved for release by the U.S. Geological 
Survey (USGS). Although the software has been subjected to rigorous 
review, the USGS reserves the right to update the software as needed 
pursuant to further analysis and review. No warran

    Solving:  Stress period:     1    Time step:     1


...

    Solving:  Stress period:     5    Time step:     2


    Solving:  Stress period:     6    Time step:     1


    Solving:  Stress period:     6    Time step:     2


 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/22 21:49:00
 Elapsed run time:  7.166 Seconds
 
 Normal termination of simulation.

Success is:  True


{'frame_count': 12,
 'spatial_range_by_layer': array([74.42275295, 74.59927232, 75.33436302, 75.6239396 ]),
 'temporal_range_by_layer': array([11.25675187, 15.92204804,  8.90741356, 11.16160897]),
 'maximum_drawdown_by_layer': array([ 9.07509759, 13.77026505,  7.77760904, 11.00435675])}

## Mass balance: the first thing to check

Before reading any map, confirm the model **converged and conserves water**.
The MF6 GWF listing budget gives the volumetric in/out by component and the
percent discrepancy. **What to look for:** a percent discrepancy near zero
(well under 1%), and a sensible balance — up-valley CHD inflow and areal/UZF
recharge in; GHB outflow at the mouth, stream and drain seepage, and pumping out.

In [3]:
import flopy
import pandas as pd

gwf_lst = next(p for p in Path(model.gwf.model_ws).glob('*.lst') if p.name != 'mfsim.lst')
incremental, _cumulative = flopy.utils.Mf6ListBudget(str(gwf_lst)).get_dataframes()
final = incremental.iloc[-1]

inflows = final[[c for c in final.index if c.endswith('_IN') and final[c] != 0]].sort_values(ascending=False)
outflows = final[[c for c in final.index if c.endswith('_OUT') and final[c] != 0]].sort_values(ascending=False)
budget = pd.concat([inflows.rename('volume'), outflows.rename('volume')]).to_frame()
print(f"percent discrepancy (final step): {final['PERCENT_DISCREPANCY']:.4f} %")
assert abs(final['PERCENT_DISCREPANCY']) < 1.0
budget

percent discrepancy (final step): 0.0000 %


,volume
TOTAL_IN,148101.046875
CHD_IN,69761.742188
LAK_IN,44225.367188
UZF-GWRCH_IN,20991.203125
STO-SY_IN,5677.370117
GHB_IN,3337.601807
SFR_IN,2520.337402
RCH_IN,1561.307861
STO-SS_IN,26.124800
TOTAL_OUT,148101.000000


## Is it physically sensible?

Two signatures define this valley and recur throughout the set:
1. The stream is mostly **gaining** in the headwaters and turns **losing** toward
   the lake — so `gaining_reach_count` should dominate but `losing_reach_count`
   is nonzero.
2. The terminal lake **perches**: its stage sits above its bottom and above the
   downgradient water table, so it leaks downward to the aquifer (shown in 02).

In [4]:
import pandas as pd

sfr = mf.canonical_sfr_signals(model)
lake_stage = model.targets.lake_stage.simulated_series()['valley_lake']
pd.Series({
    'reaches': sfr['reach_count'],
    'gaining_reaches': sfr['gaining_reach_count'],
    'losing_reaches': sfr['losing_reach_count'],
    'min_stream_depth_ft': round(sfr['minimum_depth'], 3),
    'lake_stage_ft': round(float(lake_stage.iloc[-1]), 2),
    'lake_bottom_ft': 96.0,
}, name='valley signals')

reaches                130.000
gaining_reaches        103.000
losing_reaches          27.000
min_stream_depth_ft      0.045
lake_stage_ft          104.720
lake_bottom_ft          96.000
Name: valley signals, dtype: float64

## Next

The model builds, runs, conserves water, and behaves like a real valley.
Continue to **01 · Packages and Observations** to see how each physical feature
becomes a named, queryable observation that both the plots and PEST consume.